# 03 · Chinese MentalBERT (Zhai et al., Findings of ACL 2024)

Domain-adaptive pre-training (DAPT) of `hfl/chinese-bert-wwm-ext` on MATCHA training users with
lexicon-guided whole-word masking, then fine-tuning per task. Deviations from the paper:
`docs/BASELINES.md` §1.

| Model | What it is |
|---|---|
| `matcha-depression` | the recipe on MATCHA with the paper's depression lexicon (the reproduction) |
| `matcha-random` | the same DAPT with random whole-word masking (control: the lexicon's effect) |
| `chinese-mentalbert` | the released model, fine-tuned the same way |
| `bert-wwm-ext` | the start model, no DAPT |

| | |
|---|---|
| Hardware | one GPU with 16 GB (T4 or better) |
| Time (measured on T4s) | DAPT 3.6 h per variant on two T4s (about twice that on one); MATCHA fine-tuning ~13 GPU-h (48 runs); sanity tasks ~11 h on two T4s; TF-IDF a few CPU minutes |
| Needs | notebooks 01 and 02; internet for the Hugging Face models |
| Writes | `data/dapt/`, `outputs/dapt/<variant>/`, `outputs/finetune/<task>/<model>/seed<k>/` |

Every step skips finished work and resumes after an interruption, so cells can be re-run.
The same work in one command: `python scripts/run_pipeline.py --no-limit --config configs/pipeline_mentalbert.yaml`.

> **Data use.** MATCHA is under a Data Use Agreement. These notebooks print counts and scores only, never posts; keep it that way if you add cells, and do not commit notebooks with outputs.

In [ ]:
import sys
from pathlib import Path

# make `matcha_bench` importable from a fresh clone (not needed after `pip install -e .`)
sys.path.insert(0, str((Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "src"))
from matcha_bench.notebook import check, setup, step

ROOT = setup()  # every step runs from the repository root
print(ROOT)

## 1. The DAPT corpus

Training users' posts only, segmented with jieba (primed with every lexicon, so all variants share
word boundaries), tokenised and packed into 128-token chunks once for every variant.

In [ ]:
step("scripts/build_dapt_corpus.py")

## 2. Pre-training

3 epochs, effective batch 128, lr 5e-5, 20% masking (as in the paper). The smoke run checks the
setup with a tiny model on CPU in seconds. Interrupted runs resume from their last checkpoint.

In [ ]:
step("scripts/run_dapt.py", "--config", "configs/dapt/smoke.yaml")
step("scripts/run_dapt.py", "--config", "configs/dapt/random.yaml")
step("scripts/run_dapt.py", "--config", "configs/dapt/depression.yaml")

In [ ]:
import json

for variant, eval_loss in [("random", 2.842), ("depression", 2.785)]:
    info = json.loads(Path(f"outputs/dapt/{variant}/final/run_info.json").read_text(encoding="utf-8"))
    check(f"{variant}: training chunks", info["train_chunks"], 281_264)
    if "eval_loss_true" in info:
        check(f"{variant}: eval loss", round(info["eval_loss_true"], 3), eval_loss, tol=0.03)
info = json.loads(Path("outputs/dapt/depression/final/run_info.json").read_text(encoding="utf-8"))
check("share of masked tokens from the lexicon", round(info["coverage"]["share_of_masked_tokens_from_lexicon"], 3),
      0.103, tol=0.005)

## 3. Fine-tuning on the MATCHA tasks

lr 2e-5, batch 16, ≤ 10 epochs (≤ 6 on the user tasks), best dev epoch, early stopping, balanced
class weights on the imbalanced tasks (`configs/finetune.yaml`); seeds 42, 43, 44. Plus the
TF-IDF + logistic regression reference (CPU).

In [ ]:
MATCHA_TASKS = ["matcha_disorder_post", "matcha_disorder_post_masked",
                "matcha_user_binary_masked", "matcha_user_multiclass_masked"]
MODELS = ["bert-wwm-ext", "chinese-mentalbert", "matcha-random", "matcha-depression"]

step("scripts/run_finetune.py", "--tasks", *MATCHA_TASKS, "--models", *MODELS)
step("scripts/run_tfidf.py")

## 4. Sanity check on the paper's own tasks

Our reported sanity runs had an effective batch of 32 (one run across two GPUs);
`defaults.grad_accum=2` gives the same on one GPU. EWECT-usual was stopped after 7 of 18 runs to
save GPU time; add `ewect_usual` to the task list to run it (~30 min per run).

In [ ]:
SANITY_TASKS = ["suicide", "cognitive", "waimai", "ewect_epidemic"]
step("scripts/run_finetune.py", "--tasks", *SANITY_TASKS, "--models", *MODELS, "bert-base-chinese",
     "--set", "defaults.grad_accum=2")

## 5. Scores so far

Test F1 (%), mean ± SD over seeds. The paper tables and the bootstrap come from notebook 06.

**Reported, MATCHA tasks (macro-F1):**

| Model | disorder_post | disorder_post_masked | user_binary_masked | user_multiclass_masked |
|---|---|---|---|---|
| TF-IDF + LR | 48.83 | 39.56 | 90.14 | 41.68 |
| `bert-wwm-ext` | 51.57 ± 0.84 | 42.31 ± 1.49 | 91.64 ± 0.60 | 49.25 ± 1.51 |
| `chinese-mentalbert` | 51.62 ± 3.34 | 43.70 ± 1.98 | 90.48 ± 0.88 | 50.26 ± 1.02 |
| `matcha-random` | 51.51 ± 2.62 | 45.23 ± 1.94 | 91.49 ± 1.02 | 51.49 ± 0.98 |
| `matcha-depression` | 51.26 ± 2.06 | 44.87 ± 1.88 | 92.08 ± 0.73 | 52.27 ± 0.35 |

**Reported, sanity tasks (F1; Cognitive micro, EWECT macro), ours vs the paper (Tables 3–4):**

| Model | Suicide | Cognitive | Waimai | EWECT-epidemic |
|---|---|---|---|---|
| `bert-base-chinese` | 79.84 (paper 83.46) | 71.89 (73.06) | 84.51 (87.58) | 65.15 (64.12) |
| `bert-wwm-ext` | 80.62 (84.39) | 73.68 (72.66) | 84.78 (87.44) | 65.86 (63.82) |
| `chinese-mentalbert` | 81.52 (86.15) | 73.90 (74.75) | 85.52 (89.70) | 65.86 (67.77) |
| `matcha-random` | 81.64 | 74.31 | 84.22 | 65.81 |
| `matcha-depression` | 81.92 | 74.59 | 84.38 | 67.10 |

Seed-to-seed spread on `disorder_post` is 2–3 points, so single reruns can differ by that much.

In [ ]:
from matcha_bench.stats import collect_results, summarise

print(summarise(collect_results(Path("outputs/finetune"))).to_string())

Next: `04_mentalglm.ipynb`.